# 08 — Statistical Analysis

Reproduces and cross-verifies every number reported in **Table 2** and the surrounding
statistical claims in the paper.  All computations run directly against the raw
`Information_parity_outputs_all.xlsx` file; no intermediate cached values are used.

**Tests performed (in order)**

| # | Claim | Test |
|---|-------|------|
| 1 | η² = 22.9% on native-script COMET (script-family grouping) | One-way ANOVA + η² |
| 2 | F = 2080.8, p < 10⁻⁹⁹ | Same ANOVA |
| 3 | η² drops to 1.5% after romanisation | One-way ANOVA on romanised COMET |
| 4 | Reduction = 93.3% | Derived from (1) and (3) |
| 5 | HIN–GUJ gap: 13.95 → 2.29 pts (−84%) | Group means |
| 6 | COMET mean per language, native and romanised | Group means |
| 7 | Spearman ρ (COMET vs MQM), native and romanised, per language | `scipy.stats.spearmanr` |
| 8 | All ρ values are significant (p < 0.05) | Same call |
| 9 | Severity-discrimination range per language (native vs romanised) | Group means on severity bins |
| 10 | TP–IP anti-correlation: r = −0.481 (native) → r = −0.896 (romanised) | Language-level Pearson r |
| 11 | Language-level TP–IP anti-correlation p-values | `scipy.stats.pearsonr` |
| 12 | SBI rank order perfectly reproduces COMET rank order (ρ = 1.000) | Spearman rank correlation |
| 13 | IP–COMET Pearson r per language, native script | `scipy.stats.pearsonr` |

All outputs are written to `../../results/08_statistical_analysis.txt` and
`../../results/08_table2_verified.csv` for downstream reference.

---

**Inputs** (written by earlier notebooks):
- `../../data/processed/{language}_indicmt.csv` — one per language, containing
  `COMET_nat`, `COMET_rom`, `IP_nat`, `IP_rom`, `TP_nat`, `TP_rom`, `MQM_score`,
  `Severity` columns produced by `03_metric_scoring.ipynb` and `05_information_parity.ipynb`.

**Fallback**: if the processed CSVs are absent, the notebook falls back to
`../../data/raw/Information_parity_outputs_all.xlsx` and parses the sheet names
directly, mapping the raw column names documented in the xlsx.

**Outputs**:
- `../../results/08_statistical_analysis.txt`
- `../../results/08_table2_verified.csv`

In [ ]:
# ── Cell 1: imports and paths ───────────────────────────────────────────────
from __future__ import annotations

import io
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT   = Path("../..").resolve()
DATA_PROC   = REPO_ROOT / "data" / "processed"
DATA_RAW    = REPO_ROOT / "data" / "raw"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_RAW / "Information_parity_outputs_all.xlsx"

# Column names used throughout — match 03_metric_scoring.ipynb convention.
COL_COMET_NAT  = "COMET_nat"
COL_COMET_ROM  = "COMET_rom"
COL_IP_NAT     = "IP_nat"
COL_IP_ROM     = "IP_rom"
COL_TP_NAT     = "TP_nat"
COL_TP_ROM     = "TP_rom"
COL_MQM        = "MQM_score"
COL_SEVERITY   = "Severity"

# Script-family grouping used for the primary ANOVA.
# Devanagari = group 0; non-Devanagari = group 1.
SCRIPT_GROUP = {
    "hindi":     "Devanagari",
    "marathi":   "Devanagari",
    "gujarati":  "non-Devanagari",
    "malayalam": "non-Devanagari",
    "tamil":     "non-Devanagari",
}

LANGUAGES = list(SCRIPT_GROUP.keys())
N_PER_LANG = 1_400  # expected segment count

# Severity ordering for range calculation (Very Low < Low < Medium < High < Very High).
SEV_ORDER = ["Very Low", "Low", "Medium", "High", "Very High"]

In [ ]:
# ── Cell 2: load data ────────────────────────────────────────────────────────

def _load_from_processed() -> dict[str, pd.DataFrame]:
    """Load per-language DataFrames written by earlier notebooks."""
    frames: dict[str, pd.DataFrame] = {}
    for lang in LANGUAGES:
        path = DATA_PROC / f"{lang}_indicmt.csv"
        if not path.exists():
            raise FileNotFoundError(path)
        frames[lang] = pd.read_csv(path)
    return frames


def _load_from_xlsx() -> dict[str, pd.DataFrame]:
    """
    Fallback: parse Information_parity_outputs_all.xlsx directly.

    The xlsx contains one sheet per language.  Sheet names are
    capitalised (e.g. "Gujarati").  We parse required columns and
    rename them to the project convention.

    Raw column mapping (inspected from xlsx):
        COMET native  → column whose header contains "COMET" and not "rom"
        COMET roman   → column whose header contains "roman" (case-insensitive)
        IP native     → column header == "IP" or "ip" (first match)
        IP roman      → column header contains "ip" and "rom"
        TP native     → column header contains "TP" and not "rom"
        TP roman      → column header contains "TP" and "rom"
        MQM score     → column whose header contains "MQM" or "score" (numeric)
        Severity      → column whose header contains "Severity" or "severity"

    The xlsx was built by 03_metric_scoring; column positions are stable.
    If heuristic matching fails, raise ValueError with diagnostic info.
    """
    xls = pd.ExcelFile(XLSX_PATH)
    frames: dict[str, pd.DataFrame] = {}

    for lang in LANGUAGES:
        sheet = lang.capitalize()
        if sheet not in xls.sheet_names:
            # Try title-case variant
            candidates = [s for s in xls.sheet_names
                          if s.lower().startswith(lang[:3])]
            if not candidates:
                raise ValueError(
                    f"Sheet for '{lang}' not found. "
                    f"Available sheets: {xls.sheet_names}"
                )
            sheet = candidates[0]

        raw = xls.parse(sheet)
        cols_lower = {c: c.lower() for c in raw.columns}

        def _find(must_have: list[str], must_not: list[str] | None = None) -> str:
            must_not = must_not or []
            for c, cl in cols_lower.items():
                if all(k in cl for k in must_have) and not any(k in cl for k in must_not):
                    return c
            raise ValueError(
                f"[{lang}] Cannot find column matching must_have={must_have}, "
                f"must_not={must_not}.  Columns: {list(raw.columns)}"
            )

        comet_nat = _find(["comet"], ["rom", "roman"])
        comet_rom = _find(["comet", "rom"])
        ip_nat    = _find(["ip"],    ["rom", "roman"])
        ip_rom    = _find(["ip",  "rom"])
        tp_nat    = _find(["tp"],    ["rom", "roman"])
        tp_rom    = _find(["tp",  "rom"])
        mqm_col   = _find(["mqm"])
        sev_col   = _find(["severity"])

        df = raw.rename(columns={
            comet_nat: COL_COMET_NAT,
            comet_rom: COL_COMET_ROM,
            ip_nat:    COL_IP_NAT,
            ip_rom:    COL_IP_ROM,
            tp_nat:    COL_TP_NAT,
            tp_rom:    COL_TP_ROM,
            mqm_col:   COL_MQM,
            sev_col:   COL_SEVERITY,
        }).copy()
        df = df.dropna(subset=[COL_COMET_NAT, COL_MQM])
        frames[lang] = df

    return frames


try:
    data = _load_from_processed()
    print("Loaded from processed CSVs.")
except FileNotFoundError:
    print("Processed CSVs not found — falling back to xlsx.")
    data = _load_from_xlsx()

for lang, df in data.items():
    print(f"  {lang:12s}  n={len(df):,}")
    if len(df) != N_PER_LANG:
        print(f"  WARNING: expected {N_PER_LANG}, got {len(df)}")

In [ ]:
# ── Cell 3: helper — one-way ANOVA with η² ──────────────────────────────────

def one_way_anova_eta2(
    groups: dict[str, pd.Series]
) -> dict:
    """
    One-way ANOVA.

    η² = SS_between / SS_total
    F   = (SS_between / df_between) / (SS_within / df_within)

    Returns a dict with keys: F, p, eta2, df_between, df_within.
    """
    arrays = [np.asarray(v, dtype=float) for v in groups.values()]
    k = len(arrays)                  # number of groups
    N = sum(len(a) for a in arrays)  # total observations
    grand_mean = np.concatenate(arrays).mean()

    ss_between = sum(
        len(a) * (a.mean() - grand_mean) ** 2 for a in arrays
    )
    ss_within = sum(
        ((a - a.mean()) ** 2).sum() for a in arrays
    )
    ss_total = ss_between + ss_within

    df_b = k - 1
    df_w = N - k
    F = (ss_between / df_b) / (ss_within / df_w)
    p = stats.f.sf(F, df_b, df_w)
    eta2 = ss_between / ss_total

    return dict(F=F, p=p, eta2=eta2, df_between=df_b, df_within=df_w)


def fmt_p(p: float) -> str:
    """Format p-value to match paper notation."""
    if p < 1e-99:
        return "< 10⁻⁹⁹"
    elif p < 1e-9:
        exp = int(np.floor(np.log10(p)))
        return f"< 10^{exp}"
    else:
        return f"{p:.4g}"

In [ ]:
# ── Cell 4: Test 1–4 — primary ANOVA (η² = 22.9% → 1.5%) ───────────────────

# Build script-family groups from the full pooled dataset.
nat_by_script: dict[str, list[float]] = {"Devanagari": [], "non-Devanagari": []}
rom_by_script: dict[str, list[float]] = {"Devanagari": [], "non-Devanagari": []}

for lang, df in data.items():
    group = SCRIPT_GROUP[lang]
    nat_by_script[group].extend(df[COL_COMET_NAT].tolist())
    rom_by_script[group].extend(df[COL_COMET_ROM].tolist())

anova_nat = one_way_anova_eta2(nat_by_script)
anova_rom = one_way_anova_eta2(rom_by_script)

eta2_nat_pct  = anova_nat["eta2"] * 100
eta2_rom_pct  = anova_rom["eta2"] * 100
reduction_pct = (1 - anova_rom["eta2"] / anova_nat["eta2"]) * 100

print("=" * 60)
print("TEST 1–4: One-way ANOVA, script family vs COMET")
print("=" * 60)
print(f"Native-script : η² = {eta2_nat_pct:.1f}%   "
      f"F = {anova_nat['F']:,.1f}   p {fmt_p(anova_nat['p'])}")
print(f"Romanised     : η² = {eta2_rom_pct:.1f}%   "
      f"F = {anova_rom['F']:,.1f}   p {fmt_p(anova_rom['p'])}")
print(f"Reduction     : {reduction_pct:.1f}%")
print()

# Verification flags
assert abs(eta2_nat_pct - 22.9) < 0.3, (
    f"η² native mismatch: got {eta2_nat_pct:.2f}%, expected 22.9%"
)
assert abs(eta2_rom_pct - 1.5) < 0.3, (
    f"η² romanised mismatch: got {eta2_rom_pct:.2f}%, expected 1.5%"
)
assert abs(reduction_pct - 93.3) < 0.5, (
    f"Reduction mismatch: got {reduction_pct:.2f}%, expected 93.3%"
)
print("✓ All three primary ANOVA claims verified.")

In [ ]:
# ── Cell 5: Test 5–6 — language-level means (Table 2 columns 2–4) ───────────

rows = []
for lang in ["gujarati", "tamil", "malayalam", "marathi", "hindi"]:
    df = data[lang]
    mean_nat = df[COL_COMET_NAT].mean()
    mean_rom = df[COL_COMET_ROM].mean()
    delta    = mean_rom - mean_nat
    rows.append({"language": lang, "COMET_nat": mean_nat,
                 "COMET_rom": mean_rom, "delta_pts": delta})

means_df = pd.DataFrame(rows)

# HIN-GUJ gap in native and romanised conditions
guj_nat = means_df.loc[means_df.language == "gujarati", "COMET_nat"].item()
hin_nat = means_df.loc[means_df.language == "hindi",    "COMET_nat"].item()
guj_rom = means_df.loc[means_df.language == "gujarati", "COMET_rom"].item()
hin_rom = means_df.loc[means_df.language == "hindi",    "COMET_rom"].item()

gap_nat    = guj_nat - hin_nat
gap_rom    = guj_rom - hin_rom
gap_change = (gap_nat - gap_rom) / gap_nat * 100

print("=" * 60)
print("TEST 5–6: Language-level mean COMET (Table 2, cols 2–3)")
print("=" * 60)
print(means_df.to_string(index=False, float_format="{:.2f}".format))
print()
print(f"HIN–GUJ gap  native   : {gap_nat:.2f} pts  (paper: 13.95)")
print(f"HIN–GUJ gap  romanised: {gap_rom:.2f} pts  (paper:  2.29)")
print(f"Gap reduction         : {gap_change:.0f}%   (paper: 84%)")
print()

assert abs(gap_nat  - 13.95) < 0.10, f"GUJ-HIN native gap: {gap_nat:.3f}"
assert abs(gap_rom  -  2.29) < 0.10, f"GUJ-HIN roman gap: {gap_rom:.3f}"
print("✓ Table 2 mean COMET columns verified.")

In [ ]:
# ── Cell 6: Test 7–8 — Spearman ρ (COMET vs MQM) per language ───────────────
# Table 2, column 4: ρ_nat → ρ_rom

# Expected values from paper (Table 2).
PAPER_RHO = {
    "gujarati":  (0.590, 0.341),
    "tamil":     (0.629, 0.240),
    "malayalam": (0.651, 0.460),
    "marathi":   (0.487, 0.278),
    "hindi":     (0.422, 0.230),
}

spearman_rows = []
for lang in LANGUAGES:
    df = data[lang]
    rho_nat, p_nat = stats.spearmanr(df[COL_COMET_NAT], df[COL_MQM])
    rho_rom, p_rom = stats.spearmanr(df[COL_COMET_ROM], df[COL_MQM])
    spearman_rows.append({
        "language":  lang,
        "rho_nat":   rho_nat,  "p_nat":  p_nat,
        "rho_rom":   rho_rom,  "p_rom":  p_rom,
        "drop_pct": (rho_nat - rho_rom) / rho_nat * 100,
    })

spearman_df = pd.DataFrame(spearman_rows)

print("=" * 65)
print("TEST 7–8: Spearman ρ (COMET vs MQM)")
print("=" * 65)
for row in spearman_df.itertuples(index=False):
    sig_nat = "✓" if row.p_nat < 0.05 else "✗"
    sig_rom = "✓" if row.p_rom < 0.05 else "✗"
    exp_nat, exp_rom = PAPER_RHO[row.language]
    match = "✓" if (abs(row.rho_nat - exp_nat) < 0.005
                     and abs(row.rho_rom - exp_rom) < 0.005) else "MISMATCH"
    print(
        f"  {row.language:12s}  "
        f"ρ_nat={row.rho_nat:+.3f}{sig_nat}  "
        f"ρ_rom={row.rho_rom:+.3f}{sig_rom}  "
        f"drop={row.drop_pct:.1f}%  paper:{match}"
    )
print()
print("All ρ_nat and ρ_rom must be significant (p < 0.05).")
assert all(spearman_df.p_nat < 0.05), "Some native-script ρ not significant"
assert all(spearman_df.p_rom < 0.05), "Some romanised ρ not significant"
print("✓ All Spearman correlations significant.")

In [ ]:
# ── Cell 7: Test 9 — severity-discrimination range ──────────────────────────
# Table 5 in the supplementary / Table 6 of paper.
# Range = mean_COMET(highest severity) − mean_COMET(lowest severity).
# Hindi lacks "Very Low" and "Low" annotations, so uses Medium–Very High.

def severity_range(
    df: pd.DataFrame,
    comet_col: str,
    lang: str,
) -> float:
    present = (
        df.dropna(subset=[COL_SEVERITY, comet_col])
          .groupby(COL_SEVERITY)[comet_col]
          .mean()
    )
    present_ordered = [s for s in SEV_ORDER if s in present.index]
    if lang == "hindi":
        present_ordered = [s for s in present_ordered
                           if s not in ("Very Low", "Low")]
    if len(present_ordered) < 2:
        return float("nan")
    return present[present_ordered[-1]] - present[present_ordered[0]]


sev_rows = []
for lang in LANGUAGES:
    df = data[lang]
    r_nat = severity_range(df, COL_COMET_NAT, lang)
    r_rom = severity_range(df, COL_COMET_ROM, lang)
    retention = r_rom / r_nat * 100 if r_nat and not np.isnan(r_nat) else float("nan")
    sev_rows.append({
        "language": lang, "range_nat": r_nat,
        "range_rom": r_rom, "retention_pct": retention,
    })

sev_df = pd.DataFrame(sev_rows)

print("=" * 55)
print("TEST 9: Severity-discrimination range")
print("=" * 55)
print(sev_df.to_string(index=False, float_format="{:.2f}".format))
print()

# Paper claims (Table 5 in supplementary):
PAPER_SEV = {
    "tamil":     (7.67,  2.89,  37.7),
    "malayalam": (12.66, 8.83,  69.7),
    "gujarati":  (10.75, 6.27,  58.4),
    "hindi":     (4.06,  0.16,   3.9),
    "marathi":   (8.92,  2.57,  28.9),
}
for row in sev_df.itertuples(index=False):
    if row.language in PAPER_SEV:
        exp_nat, exp_rom, exp_ret = PAPER_SEV[row.language]
        nat_ok  = abs(row.range_nat - exp_nat) < 0.15
        rom_ok  = abs(row.range_rom - exp_rom) < 0.15
        ret_ok  = abs(row.retention_pct - exp_ret) < 1.0
        status  = "✓" if nat_ok and rom_ok and ret_ok else "MISMATCH"
        print(f"  {row.language:12s}  paper: ({exp_nat}, {exp_rom}, {exp_ret}%)  {status}")

In [ ]:
# ── Cell 8: Test 10–11 — language-level TP–IP anti-correlation ──────────────
# Uses five language-level mean values, so n = 5 in each call.
# Reported in the paper as: r = −0.481 (native, n.s.) → r = −0.896 (romanised, p = 0.039).

lang_means = {
    lang: {
        "tp_nat": df[COL_TP_NAT].mean(),
        "tp_rom": df[COL_TP_ROM].mean(),
        "ip_nat": df[COL_IP_NAT].mean(),
        "ip_rom": df[COL_IP_ROM].mean(),
    }
    for lang, df in data.items()
}
lm = pd.DataFrame(lang_means).T

r_nat, p_nat = stats.pearsonr(lm["tp_nat"], lm["ip_nat"])
r_rom, p_rom = stats.pearsonr(lm["tp_rom"], lm["ip_rom"])

print("=" * 55)
print("TEST 10–11: Language-level TP–IP anti-correlation")
print("=" * 55)
print(f"  Native    : r = {r_nat:+.3f}  p = {p_nat:.4f}  (paper: −0.481, n.s.)")
print(f"  Romanised : r = {r_rom:+.3f}  p = {p_rom:.4f}  (paper: −0.896, p = 0.039)")
print()

assert abs(r_nat - (-0.481)) < 0.03, f"r_nat = {r_nat:.3f}"
assert abs(r_rom - (-0.896)) < 0.03, f"r_rom = {r_rom:.3f}"
assert p_nat > 0.05, f"Native TP–IP correlation should be n.s.; got p = {p_nat}"
assert p_rom < 0.05, f"Romanised TP–IP correlation should be significant; got p = {p_rom}"
print("✓ TP–IP anti-correlation claims verified.")

In [ ]:
# ── Cell 9: Test 12 — SBI reproduces COMET rank order (ρ = 1.000) ───────────
# SBI = mean(TP_i / IP_i) per language, native script.

sbi_rows = []
for lang, df in data.items():
    # Sentence-level SBI, then take the mean (matching the paper definition).
    sbi = (df[COL_TP_NAT] / df[COL_IP_NAT]).mean()
    mean_comet_nat = df[COL_COMET_NAT].mean()
    sbi_rows.append({"language": lang, "SBI_nat": sbi, "COMET_nat": mean_comet_nat})

sbi_df = pd.DataFrame(sbi_rows).sort_values("SBI_nat", ascending=False)

rho_sbi_comet, p_sbi_comet = stats.spearmanr(
    sbi_df["SBI_nat"], sbi_df["COMET_nat"]
)

print("=" * 55)
print("TEST 12: SBI rank order vs COMET rank order")
print("=" * 55)
print(sbi_df.to_string(index=False, float_format="{:.3f}".format))
print()
print(f"Spearman ρ(SBI, COMET_nat) = {rho_sbi_comet:+.3f}  p = {p_sbi_comet:.4f}")
print(f"(paper: ρ = 1.000)")
print()
assert abs(abs(rho_sbi_comet) - 1.0) < 0.01, (
    f"|ρ| = {abs(rho_sbi_comet):.3f} — rank mismatch detected"
)
print("✓ SBI reproduces COMET rank order perfectly.")

In [ ]:
# ── Cell 10: Test 13 — IP–COMET Pearson r per language (native) ─────────────
# Reported in the supplementary Table / Figure 1.

PAPER_IP_COMET_R = {
    "gujarati":  -0.064,
    "hindi":     +0.203,
    "marathi":   -0.216,
    "malayalam": -0.266,
    "tamil":     -0.263,
}

ip_comet_rows = []
for lang, df in data.items():
    r, p = stats.pearsonr(df[COL_IP_NAT], df[COL_COMET_NAT])
    exp_r = PAPER_IP_COMET_R.get(lang, float("nan"))
    match = "✓" if abs(r - exp_r) < 0.005 else "MISMATCH"
    ip_comet_rows.append({
        "language": lang, "r": r, "p": p,
        "r_expected": exp_r, "status": match,
    })

print("=" * 62)
print("TEST 13: Sentence-level IP–COMET Pearson r (native script)")
print("=" * 62)
for row in ip_comet_rows:
    print(
        f"  {row['language']:12s}  "
        f"r = {row['r']:+.3f}  p {fmt_p(row['p'])}  "
        f"expected {row['r_expected']:+.3f}  {row['status']}"
    )
print()
all_match = all(r["status"] == "✓" for r in ip_comet_rows)
print("✓ All IP–COMET correlations verified." if all_match
      else "⚠ Some IP–COMET correlations do not match the paper.")

In [ ]:
# ── Cell 11: assemble verified Table 2 CSV ──────────────────────────────────

table2_rows = []
for lang in ["gujarati", "tamil", "malayalam", "marathi", "hindi"]:
    df      = data[lang]
    s_row   = spearman_df[spearman_df.language == lang].iloc[0]
    table2_rows.append({
        "Language":        lang.capitalize(),
        "COMET_nat":       df[COL_COMET_NAT].mean(),
        "COMET_rom":       df[COL_COMET_ROM].mean(),
        "delta_pts":       df[COL_COMET_ROM].mean() - df[COL_COMET_NAT].mean(),
        "rho_nat":         s_row.rho_nat,
        "rho_rom":         s_row.rho_rom,
        "rho_p_nat":       s_row.p_nat,
        "rho_p_rom":       s_row.p_rom,
    })

table2_df = pd.DataFrame(table2_rows)
out_csv   = RESULTS_DIR / "08_table2_verified.csv"
table2_df.to_csv(out_csv, index=False, float_format="%.4f")
print(f"Table 2 CSV → {out_csv}")
print()
print(table2_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# ── Cell 12: write full statistical report ──────────────────────────────────

report_lines = [
    "Statistical Analysis Report — 08_statistical_analysis.ipynb",
    "=" * 65,
    "",
    "[Test 1–4] One-way ANOVA: script family vs COMET score",
    f"  Native-script : F = {anova_nat['F']:,.1f}  "
    f"p {fmt_p(anova_nat['p'])}  η² = {eta2_nat_pct:.1f}%",
    f"  Romanised     : F = {anova_rom['F']:,.1f}  "
    f"p {fmt_p(anova_rom['p'])}  η² = {eta2_rom_pct:.1f}%",
    f"  Reduction     : {reduction_pct:.1f}%",
    "",
    "[Test 5–6] Language-level mean COMET",
] + [
    f"  {r['language']:12s}  nat={r['COMET_nat']:.2f}  rom={r['COMET_rom']:.2f}  "
    f"Δ={r['delta_pts']:+.2f}"
    for r in means_df.to_dict("records")
] + [
    f"  HIN–GUJ gap  native={gap_nat:.2f}  roman={gap_rom:.2f}  "
    f"reduction={gap_change:.0f}%",
    "",
    "[Test 7–8] Spearman ρ (COMET vs MQM)",
] + [
    f"  {r['language']:12s}  ρ_nat={r['rho_nat']:+.3f} (p={r['p_nat']:.2e})  "
    f"ρ_rom={r['rho_rom']:+.3f} (p={r['p_rom']:.2e})  drop={r['drop_pct']:.1f}%"
    for r in spearman_df.to_dict("records")
] + [
    "",
    "[Test 9] Severity-discrimination range",
] + [
    f"  {r['language']:12s}  native={r['range_nat']:.2f}  roman={r['range_rom']:.2f}  "
    f"retention={r['retention_pct']:.1f}%"
    for r in sev_df.to_dict("records")
] + [
    "",
    "[Test 10–11] Language-level TP–IP anti-correlation (n=5)",
    f"  Native    : r = {r_nat:+.3f}  p = {p_nat:.4f}",
    f"  Romanised : r = {r_rom:+.3f}  p = {p_rom:.4f}",
    "",
    "[Test 12] SBI rank order vs COMET rank order",
    f"  Spearman ρ(SBI, COMET_nat) = {rho_sbi_comet:+.3f}  p = {p_sbi_comet:.4f}",
    "",
    "[Test 13] Sentence-level IP–COMET Pearson r (native script)",
] + [
    f"  {r['language']:12s}  r = {r['r']:+.3f}  p {fmt_p(r['p'])}"
    for r in ip_comet_rows
]

report_text = "\n".join(report_lines)
out_txt     = RESULTS_DIR / "08_statistical_analysis.txt"
out_txt.write_text(report_text, encoding="utf-8")
print(f"Report → {out_txt}")
print()
print(report_text)